In [1]:
import json
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Setup paths ---
root = Path(__file__).resolve().parent
output_dir = root / "outputs" / "w3d3_decision_trees"
output_dir.mkdir(parents=True, exist_ok=True)

# --- Load dataset ---
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Train model ---
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)

# --- Evaluate ---
y_pred = clf.predict(X_test)
summary = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
}

# Save metrics locally
(output_dir / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

# --- Fix MLflow backend ---
# Use SQLite instead of filesystem store
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# --- Log experiment ---
mlflow.set_experiment("w3d3_decision_trees")
with mlflow.start_run(run_name="tree-and-forest-tuning"):
    mlflow.log_params({"max_depth": 4, "random_state": 42})
    mlflow.log_metrics(summary)
    mlflow.sklearn.log_model(clf, "decision_tree_model")

print("Experiment completed. Metrics:", summary)


NameError: name '__file__' is not defined